# Unit 2 — Q-learning with FrozenLake and Taxi

Implement tabular Q-learning from scratch, train a policy for each environment, and evaluate on separate episodes. The Taxi evaluation uses the 100 start seeds from the original course notebook.

Run the cells from top to bottom. The first cell installs dependencies, and the training cells regenerate both checkpoints and replays. The final section shows how to publish the models.

[Hugging Face Deep RL Course — Unit 2 hands-on](https://huggingface.co/learn/deep-rl-course/unit2/hands-on)

In [ ]:
%pip -q install "gymnasium[toy-text]==0.29.1" "imageio[ffmpeg]" "huggingface_hub"

## Q-table and policy

Each Q-table row is a state and each column is an action. The table starts at zero. During training, the agent samples a random action with probability epsilon and otherwise chooses the action with the largest Q-value. Epsilon decays over the training episodes.

The update moves the Q-value toward the observed reward plus the discounted best Q-value of the next state. When the environment terminates, the target is the reward alone. A time limit ends the rollout while retaining the bootstrap term.

In [1]:
import os
from pathlib import Path
import pickle
import json
import hashlib

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

import gymnasium as gym
import imageio.v2 as imageio
import numpy as np

def greedy_policy(qtable, state):
    return int(np.argmax(qtable[state]))

def epsilon_greedy_policy(qtable, state, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(qtable.shape[1]))
    return greedy_policy(qtable, state)

def train(spec):
    rng = np.random.default_rng(spec["train_seed"])
    with gym.make(spec["env_id"], **spec["env_kwargs"]) as env:
        qtable = np.zeros((env.observation_space.n, env.action_space.n))
        for episode in range(spec["n_training_episodes"]):
            epsilon = spec["min_epsilon"] + (spec["max_epsilon"] - spec["min_epsilon"]) * np.exp(-spec["decay_rate"] * episode)
            state, _ = env.reset(seed=spec["train_seed"] if episode == 0 else None)
            for _ in range(spec["max_steps"]):
                action = epsilon_greedy_policy(qtable, state, epsilon, rng)
                next_state, reward, terminated, truncated, _ = env.step(action)
                target = reward if terminated else reward + spec["gamma"] * np.max(qtable[next_state])
                qtable[state, action] += spec["learning_rate"] * (target - qtable[state, action])
                if terminated or truncated:
                    break
                state = next_state
    return qtable

def run_episode(env, qtable, seed, max_steps, record=False):
    state, _ = env.reset(seed=seed)
    total = 0.0
    frames = [env.render()] if record else []
    for _ in range(max_steps):
        state, reward, terminated, truncated, _ = env.step(greedy_policy(qtable, state))
        total += reward
        if record:
            frames.append(env.render())
        if terminated or truncated:
            break
    return total, frames

def evaluate(spec, qtable):
    with gym.make(spec["env_id"], **spec["env_kwargs"]) as env:
        returns = [run_episode(env, qtable, seed, spec["max_steps"])[0] for seed in spec["eval_seed"]]
    return np.asarray(returns, dtype=float)

## Inspect the environments and train

FrozenLake uses the deterministic 4×4 map. Taxi uses its standard discrete action and observation spaces. The training seed is fixed to reproduce the Q-tables.

In [2]:
SPECS = {'q-FrozenLake-v1-4x4-noSlippery': {'max_steps': 99, 'gamma': 0.95, 'learning_rate': 0.7, 'max_epsilon': 1.0, 'min_epsilon': 0.05, 'train_seed': 42, 'n_eval_episodes': 100, 'env_id': 'FrozenLake-v1', 'env_kwargs': {'map_name': '4x4', 'is_slippery': False}, 'eval_seed': [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099], 'n_training_episodes': 10000, 'decay_rate': 0.0005}, 'q-Taxi-v3': {'max_steps': 99, 'gamma': 0.95, 'learning_rate': 0.7, 'max_epsilon': 1.0, 'min_epsilon': 0.05, 'train_seed': 42, 'n_eval_episodes': 100, 'env_id': 'Taxi-v3', 'env_kwargs': {}, 'eval_seed': [16, 54, 165, 177, 191, 191, 120, 80, 149, 178, 48, 38, 6, 125, 174, 73, 50, 172, 100, 148, 146, 6, 25, 40, 68, 148, 49, 167, 9, 97, 164, 176, 61, 7, 54, 55, 161, 131, 184, 51, 170, 12, 120, 113, 95, 126, 51, 98, 36, 135, 54, 82, 45, 95, 89, 59, 95, 124, 9, 113, 58, 85, 51, 134, 121, 169, 105, 21, 30, 11, 50, 65, 12, 43, 82, 145, 152, 97, 106, 55, 31, 85, 38, 112, 102, 168, 123, 97, 21, 83, 158, 26, 80, 63, 5, 81, 32, 11, 28, 148], 'n_training_episodes': 25000, 'decay_rate': 0.005}}

In [3]:
qtables = {}
for name, spec in SPECS.items():
    with gym.make(spec["env_id"], **spec["env_kwargs"]) as env:
        state, _ = env.reset(seed=spec["train_seed"])
        action = 0
        next_state, reward, terminated, truncated, _ = env.step(action)
        print(f"{name}: {env.observation_space.n} states, {env.action_space.n} actions")
        print(f"Example transition: {state} -> {next_state}, action {action}, reward {reward:g}")
    qtables[name] = train(spec)
    print(f"Trained on {spec['n_training_episodes']:,} episodes; table shape {qtables[name].shape}")

q-FrozenLake-v1-4x4-noSlippery: 16 states, 4 actions
Example transition: 0 -> 0, action 0, reward 0
Trained on 10,000 episodes; table shape (16, 4)
q-Taxi-v3: 500 states, 6 actions
Example transition: 386 -> 486, action 0, reward -1
Trained on 25,000 episodes; table shape (500, 6)


## Evaluate the greedy policies

Each policy is evaluated over 100 episodes without exploration. The course score is the mean episode reward minus the population standard deviation. The course hands-on specifies 4.5 for Taxi-v3.

In [4]:
for name, spec in SPECS.items():
    rewards = evaluate(spec, qtables[name])
    mean, std = rewards.mean(), rewards.std()
    print(f"{name} | mean {mean:.2f} | std {std:.2f} | score {mean-std:.2f}")
    if spec["env_id"] == "Taxi-v3":
        print(f"Taxi meets the 4.5 threshold: {mean - std >= 4.5}")

q-FrozenLake-v1-4x4-noSlippery | mean 1.00 | std 0.00 | score 1.00
q-Taxi-v3 | mean 7.50 | std 2.71 | score 4.79
Taxi meets the 4.5 threshold: True


## Export and record replays

The first evaluation episode is recorded for each agent. The model card contains the metadata used by the course tracker.

In [ ]:
for name, spec in SPECS.items():
    root = Path("unit2_outputs") / name
    root.mkdir(parents=True, exist_ok=True)
    qtable = qtables[name]
    returns = evaluate(spec, qtable)
    mean, std = float(returns.mean()), float(returns.std())
    result = dict(env_id=spec["env_id"], mean_reward=mean, std_reward=std, score=mean-std,
                  n_eval_episodes=len(returns), deterministic_policy=True, episode_rewards=returns.tolist())
    payload = dict(spec, qtable=qtable)
    if spec["env_kwargs"]:
        payload.update(map_name="4x4", slippery=False)
    with (root / "q-learning.pkl").open("wb") as handle:
        pickle.dump(payload, handle, protocol=4)
    with (root / "q-learning.pkl").open("rb") as handle:
        np.testing.assert_array_equal(evaluate(spec, pickle.load(handle)["qtable"]), returns)
    (root / "results.json").write_text(json.dumps(result,indent=2)+"\n")
    with gym.make(spec["env_id"], **spec["env_kwargs"], render_mode="rgb_array") as env:
        replay_reward, frames = run_episode(env, qtable, spec["eval_seed"][0], spec["max_steps"], record=True)
    assert replay_reward == returns[0]
    with imageio.get_writer(root / "replay.mp4", fps=2, codec="libx264", macro_block_size=2, ffmpeg_log_level="error") as video:
        for frame in frames:
            video.append_data(frame)
    env_tag = spec["env_id"] if spec["env_id"]=="Taxi-v3" else "FrozenLake-v1-4x4-no_slippery"
    (root / "README.md").write_text(f"""---
tags:
- {env_tag}
- q-learning
- reinforcement-learning
- custom-implementation
model-index:
- name: {name}
  results:
  - task:
      type: reinforcement-learning
      name: reinforcement-learning
    dataset:
      type: {env_tag}
      name: {env_tag}
    metrics:
    - type: mean_reward
      name: mean_reward
      value: "{mean:.2f} +/- {std:.2f}"
---

# {name}

Tabular Q-learning agent trained from scratch for the [Hugging Face Deep RL Course, Unit 2](https://huggingface.co/learn/deep-rl-course/unit2/hands-on).

- Environment: `{spec['env_id']}`
- Training: {spec['n_training_episodes']:,} episodes, seed {spec['train_seed']}
- Q-table: {qtable.shape[0]} states × {qtable.shape[1]} actions
- Evaluation: {len(returns)} episodes using the greedy policy
- Mean reward: {mean:.2f}
- Population standard deviation: {std:.2f}
- Course score (mean minus standard deviation): **{mean-std:.2f}**

The first evaluation episode is recorded in `replay.mp4`. The checkpoint `q-learning.pkl` contains the Q-table and settings. Use Gymnasium 0.29.1 and load pickle files only from sources you trust.
""", encoding="utf-8")
    print(name, qtable.shape, f"{mean:.6f} +/- {std:.6f}; score {mean-std:.6f}; replay {replay_reward:g}",flush=True)

In [ ]:
from IPython.display import Video, display

for name in SPECS:
    print(name)
    display(Video(str(Path("unit2_outputs") / name / "replay.mp4"), embed=True, width=450))

## Publish to Hugging Face Hub

The Taxi model is the submission for course credit. The FrozenLake model can also be shared. The optional cell below uses your Hugging Face account and creates or updates public model repositories. Enter any write token only into the official sign-in prompt.

Check [the course progress tracker](https://huggingface.co/spaces/ThomasSimonini/Check-my-progress-Deep-RL-Course) after publication.

In [ ]:
PUBLISH_TO_HUB = False

if PUBLISH_TO_HUB:
    from huggingface_hub import HfApi, notebook_login

    notebook_login(add_to_git_credential=False)
    api = HfApi()
    username = api.whoami()["name"]
    for name in SPECS:
        repo_id = f"{username}/{name}"
        api.create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
        api.upload_folder(
            repo_id=repo_id,
            folder_path=Path("unit2_outputs") / name,
            commit_message="Publish Unit 2 Q-learning model and evaluation",
        )
        print(f"https://huggingface.co/{repo_id}")

### Reload a published checkpoint

Set the ID of your published repository. Only unpickle checkpoints from repositories you trust.

In [ ]:
HUB_REPO_ID = ""

if HUB_REPO_ID:
    from huggingface_hub import hf_hub_download

    checkpoint = hf_hub_download(HUB_REPO_ID, "q-learning.pkl")
    with open(checkpoint, "rb") as handle:
        model = pickle.load(handle)
    rewards = evaluate(model, model["qtable"])
    print(f"Reloaded checkpoint score: {rewards.mean() - rewards.std():.2f}")